#  Notebook 01 — Modern Python Environments with `uv`

**LISA CosWG Computing Bootcamp 2026**

---

### What you will learn

| # | Topic |
|---|-------|
| 1 | What `uv` is and why it matters |
| 2 | How to install `uv` |
| 3 | Creating an isolated Python environment |
| 4 | Adding and removing packages |
| 5 | Understanding `pyproject.toml` and `uv.lock` |
| 6 | How to activate and use the environment |
| 7 | How this translates to your local machine |

---

###  Colab vs. Local: one honest note upfront

Google Colab already has its own Python environment, so we **cannot fully replace it** with a `uv`-managed one inside Colab. 
What we *can* do — and what this notebook focuses on — is:

- Install `uv` and run all its commands
- Create a real virtual environment and install packages into it
- Run scripts **inside** that environment using `uv run`
- Understand the workflow so you can replicate it perfectly **on your own laptop or HPC cluster**

Think of Colab here as a **sandbox to learn the commands** — the payoff is on your own machine.

---
## 1 — What is `uv` and why should you care?

`uv` is a **blazing-fast Python package and project manager**, written in Rust, developed by [Astral](https://astral.sh). It replaces a whole stack of tools you might already know:

| Old tool | What `uv` replaces |
|---|---|
| `pip` | installing packages |
| `venv` / `virtualenv` | creating environments |
| `pip-tools` | locking dependencies |
| `pyenv` | managing Python versions |
| `pipx` | installing CLI tools |

### Why is it better?

-  **10–100× faster** than `pip` (resolves and installs in seconds, not minutes)
-  **Reproducible** — every install is locked via `uv.lock`
-  **One tool** — no more juggling `conda`, `pip`, `venv`, `pyenv`
-  **Clean** — environments are isolated and easy to throw away
-  **Cross-platform** — Linux, macOS, Windows

In scientific computing (LISA data analysis, simulations, ML pipelines), reproducibility is **critical**. 
`uv` makes it easy to share an environment that works identically on your laptop, a collaborator's machine, and a cluster.

---
## 2 — Installing `uv`

### On Google Colab (this notebook)

In [ ]:
%%bash
# Install uv using the official installer
curl -LsSf https://astral.sh/uv/install.sh | sh

# Add uv to PATH for this session
export PATH="$HOME/.cargo/bin:$HOME/.local/bin:$PATH"

# Verify installation
uv --version

In [ ]:
import os

# Make uv available in all subsequent cells
os.environ["PATH"] = f"/root/.cargo/bin:/root/.local/bin:{os.environ['PATH']}"

# Confirm it is visible
import subprocess
result = subprocess.run(["uv", "--version"], capture_output=True, text=True)
print(result.stdout)

### 💻 On your local machine (macOS / Linux)

Open a terminal and run:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Then restart your terminal (or run `source ~/.bashrc` / `source ~/.zshrc`).

### 💻 On Windows (PowerShell)

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

### ✅ Check it works

```bash
uv --version
# uv 0.5.x (or later)
```

---
## 3 — Creating a Python environment

With `uv` you first **initialise a project** (creates a `pyproject.toml`) and then create the virtual environment.

```
my_project/
├── pyproject.toml   ← project metadata + dependencies
├── uv.lock          ← exact locked versions (auto-generated)
├── .venv/           ← the virtual environment (never commit this!)
└── my_script.py
```

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"

# Create a project directory
mkdir -p /tmp/lisa_demo
cd /tmp/lisa_demo

# Initialise a uv project (creates pyproject.toml)
uv init .

echo ""
echo "=== Files created ==="
ls -la

echo ""
echo "=== pyproject.toml contents ==="
cat pyproject.toml

### What is `pyproject.toml`?

This is the **single source of truth** for your project. It records:
- The project name, version, and description
- Which Python version you need
- Which packages are required (you'll add these in section 4)

It is the modern replacement for `requirements.txt` — but much more powerful.

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"
cd /tmp/lisa_demo

# Create the virtual environment (uses the Python version in pyproject.toml)
uv venv

echo ""
echo "=== .venv directory created ==="
ls .venv/

echo ""
echo "=== Python inside the venv ==="
.venv/bin/python --version

### What just happened?

`uv venv` created a `.venv/` folder containing a **completely isolated Python installation**.
Any packages you install go here — they do **not** affect your system Python or any other project.

You can also specify a particular Python version:

```bash
uv venv --python 3.11
uv venv --python 3.12
```

`uv` will download that Python version automatically if you don't have it — no `pyenv` needed!

---
## 4 — Adding packages

There are two ways to add packages with `uv`:

| Command | Effect |
|---|---|
| `uv add numpy` | Adds to `pyproject.toml` **and** installs |
| `uv pip install numpy` | Installs only (does not update `pyproject.toml`) |

**Always prefer `uv add`** — it keeps your `pyproject.toml` up to date and makes the environment reproducible.

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"
cd /tmp/lisa_demo

# Add some scientific packages — watch how fast this is!
uv add numpy scipy matplotlib

echo ""
echo "=== Updated pyproject.toml ==="
cat pyproject.toml

In [ ]:
%%bash
cd /tmp/lisa_demo

echo "=== uv.lock (first 40 lines) ==="
head -40 uv.lock

### What is `uv.lock`?

The lock file records the **exact version of every package** (including transitive dependencies — the dependencies of your dependencies).  

- ✅ **Commit `uv.lock` to git** — this is how collaborators get *exactly* the same environment
- ✅ **Commit `pyproject.toml` to git** — this is the human-readable specification  
- ❌ **Never commit `.venv/`** — it is large and machine-specific (add it to `.gitignore`)

If a collaborator clones your repo, they just run:
```bash
uv sync
```
...and they get the **identical** environment in seconds.

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"
cd /tmp/lisa_demo

# Add a specific version
uv add "astropy>=6.0"

# Add a dev-only dependency (testing, not needed in production)
uv add --dev pytest hypothesis

echo ""
echo "=== Final pyproject.toml ==="
cat pyproject.toml

### Dev dependencies

`--dev` flags a package as a **development-only dependency** — things like `pytest`, `hypothesis`, `black`, `ruff`.  
They are listed separately in `pyproject.toml` and can be excluded when deploying to production or a container.

```bash
uv sync              # installs everything (including dev)
uv sync --no-dev     # installs only production dependencies
```

---
## 5 — Removing packages and listing what is installed

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"
cd /tmp/lisa_demo

echo "=== Packages installed in the environment ==="
uv pip list

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"
cd /tmp/lisa_demo

# Remove a package — uv updates pyproject.toml and uv.lock automatically
uv remove matplotlib

echo ""
echo "=== pyproject.toml after removal ==="
cat pyproject.toml

---
## 6 — Using the environment: `source` vs `uv run`

There are two ways to use your environment:

### Method A — Activate (classic approach, works in your terminal)

```bash
# On Linux / macOS:
source .venv/bin/activate

# On Windows (PowerShell):
.venv\Scripts\Activate.ps1

# Your prompt will change to show (.venv)
(.venv) $ python my_script.py
(.venv) $ python -c "import numpy; print(numpy.__version__)"

# Deactivate when done:
deactivate
```

### Method B — `uv run` (no activation needed — great for scripts and CI)

```bash
uv run python my_script.py
uv run pytest
uv run jupyter notebook
```

**`uv run` automatically uses the `.venv` in the current directory** — no need to activate first.

### Why can't we just `source` in Colab?

Each `%%bash` cell in Colab spawns a **new shell process** — so `source .venv/bin/activate` would only last for that one cell.  
That is why we use `uv run` below, which works perfectly in Colab.

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"
cd /tmp/lisa_demo

# Write a small Python script
cat > gw_demo.py << 'EOF'
import numpy as np
from astropy import units as u
from astropy import constants as const

print("=== Gravitational Wave Strain Demo ===")
print(f"numpy version : {np.__version__}")

# A simple chirp mass calculation
m1 = 30 * u.Msun   # 30 solar masses
m2 = 25 * u.Msun   # 25 solar masses

chirp_mass = (m1 * m2)**(3/5) / (m1 + m2)**(1/5)
print(f"Chirp mass for ({m1}, {m2}) binary: {chirp_mass:.2f}")

# Simple frequency array for LISA band
f = np.logspace(-4, -1, 1000)  # 0.1 mHz to 0.1 Hz
print(f"LISA frequency band: {f[0]:.2e} Hz to {f[-1]:.2e} Hz")
print(f"Number of frequency bins: {len(f)}")
EOF

# Run it INSIDE the uv environment — no activation needed!
uv run python gw_demo.py

---
## 7 — Reproducing an environment from scratch

This is the **most important workflow** for collaboration.

Imagine a collaborator clones your repository. They have **no packages installed**.
All they need to do is:

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"
cd /tmp/lisa_demo

# Simulate: collaborator deletes the .venv
rm -rf .venv
echo "Deleted .venv — simulating a fresh clone"
echo ""

# ONE command to recreate everything exactly:
uv sync

echo ""
echo "=== Environment is back! ==="
uv run python -c "import numpy, astropy; print('numpy:', numpy.__version__, '| astropy:', astropy.__version__)"

---
## 8 — Managing Python versions with `uv`

One of `uv`'s killer features: it can **download and manage Python versions** for you.

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"

# List Python versions available to install
uv python list --only-downloads 2>/dev/null | head -15

echo ""
echo "On your local machine you can install any Python version:"
echo "  uv python install 3.11"
echo "  uv python install 3.12"
echo "  uv venv --python 3.11   # creates venv with Python 3.11"

---
## 9 — Full local workflow cheat sheet

Here is everything you need for a typical scientific project on your own machine:

```bash
# ── SETUP (once per project) ─────────────────────────────────────

mkdir my_lisa_analysis && cd my_lisa_analysis
uv init .                      # create pyproject.toml
uv venv                        # create .venv/
source .venv/bin/activate      # activate (your prompt changes)

# ── ADDING PACKAGES ───────────────────────────────────────────────

uv add numpy scipy astropy     # production deps
uv add --dev pytest hypothesis # dev-only deps
uv add "gwpy>=3.0"             # with version constraint

# ── DAILY USE ─────────────────────────────────────────────────────

source .venv/bin/activate      # activate at start of session
python my_script.py            # run your code
jupyter notebook               # or start jupyter
deactivate                     # done for the day

# ── OR use uv run (no activation needed) ─────────────────────────

uv run python my_script.py
uv run pytest
uv run jupyter notebook

# ── COLLABORATION ─────────────────────────────────────────────────

git add pyproject.toml uv.lock  # commit these
echo '.venv/' >> .gitignore      # NEVER commit .venv

# Collaborator clones repo and runs:
uv sync                          # recreates identical environment

# ── UPDATING ─────────────────────────────────────────────────────

uv add numpy --upgrade          # upgrade a specific package
uv lock --upgrade               # upgrade all packages
uv remove scipy                 # remove a package
```

---

### 📋 `.gitignore` entries for `uv` projects

```gitignore
# Python environments
.venv/
__pycache__/
*.pyc

# uv cache (optional — uv manages this itself)
.uv/
```

---
## 10 — Mini exercise 🧪

Try these steps yourself — either here in Colab or on your local machine:

1. **Create** a new project called `my_gw_project`
2. **Add** the packages `numpy`, `scipy`, and `gwpy`
3. **Write** a small script that imports all three and prints their versions
4. **Run** it with `uv run python your_script.py`
5. **Remove** `gwpy` and verify it disappears from `pyproject.toml`
6. **Commit** `pyproject.toml` and `uv.lock` to a git repo

In [ ]:
%%bash
export PATH="/root/.cargo/bin:/root/.local/bin:$PATH"

# Your code here!
# Step 1: Create the project
mkdir -p /tmp/my_gw_project
cd /tmp/my_gw_project

echo "Try completing the rest of the steps above!"
echo "Use the cheat sheet in Section 9 as your reference."

---
## Summary

| Command | What it does |
|---|---|
| `uv init .` | Create a new project with `pyproject.toml` |
| `uv venv` | Create a virtual environment in `.venv/` |
| `source .venv/bin/activate` | Activate the environment (local terminal) |
| `uv add <package>` | Install a package + update `pyproject.toml` |
| `uv add --dev <package>` | Install a dev-only package |
| `uv remove <package>` | Remove a package |
| `uv sync` | Recreate environment from `uv.lock` |
| `uv run python script.py` | Run a script inside the environment |
| `uv pip list` | List installed packages |
| `uv lock --upgrade` | Upgrade all packages |

---

## ➡️ Next: Notebook 02 — Using the OpenAI API in Python

In the next notebook we will use `uv` to install `openai`, set up our API key securely, and interact with GPT-4o directly from Python — asking it to explain and write gravitational wave analysis code.